In [8]:
import pandas as pd
import re

# Load CSV (Windows-safe path)
df = pd.read_csv(r'C:\Users\990215322\Desktop\MuProMAC\out\251110\results\FIFO_l0.28_actuator_manufacturing_with_rework.csv')

print(f"Total events: {len(df)}")
print(f"Unique cases: {df['case_id'].nunique()}")
print(f"Time range: {df['timestamp'].min():.2f} - {df['timestamp'].max():.2f}")
print(f"\nActivity distribution:\n{df['activity'].value_counts()}")

# 2) Check FIFO order at a pooled station (e.g., ASSEMBLY_2)
st = "ASSEMBLY_2"
x = df[(df["status"]=="running") & (df["activity"]==st)].copy().sort_values("timestamp")
print("\nFirst 10 starts at", st)
print(x[["case_id","timestamp","resource"]].head(10))

# 3) Check that A1 rework returns to the SAME queue/lane
# Find an A1 label that exists in this log (pooled or lane-specific)
a1_labels = [a for a in df["activity"].unique() if re.fullmatch(r"ASSEMBLY_1(_\d+)?", str(a))]
if a1_labels:
    lane = sorted(a1_labels)[0]  # pick the first matching label
    qc_label = "QC_AFTER_ASSEMBLY1" if lane == "ASSEMBLY_1" else f"QC_AFTER_{lane}"
    cid_example = df[(df["activity"]==lane) & (df["status"]=="running")]["case_id"].head(1)
    if not cid_example.empty:
        cid = int(cid_example.iloc[0])
        y = df[(df["case_id"]==cid) & df["activity"].isin([lane, qc_label])].sort_values("timestamp")
        print(f"\nTrace for case {cid} at {lane}:")
        print(y[["timestamp","activity","status","resource"]].head(20))
    else:
        print(f"\nNo runs found for {lane} in this file.")
else:
    print("\nNo ASSEMBLY_1 labels found (pooled or lanes).")

# 4) Visual FIFO check at ASSEMBLY_2 (first 15 starts)
def station_fifo_sample(station):
    return df[(df["activity"]==station) & (df["status"]=="running")] \
             [["case_id","timestamp","resource"]].sort_values("timestamp").head(15)

print("\nVisual FIFO check (first 15 at ASSEMBLY_2):")
print(station_fifo_sample("ASSEMBLY_2"))


Total events: 147934
Unique cases: 8332
Time range: 18.54 - 29999.29

Activity distribution:
activity
MOULDING              17614
ASSEMBLY_2            17509
ASSEMBLY_1            17508
SORTING               17476
PACKAGING             17434
QC_AFTER_MOULDING      8805
QC_AFTER_ASSEMBLY1     8754
QC_AFTER_ASSEMBLY2     8745
QC_AFTER_SORTING       8734
QC_AFTER_PACKAGING     8717
START                  8332
END                    8306
Name: count, dtype: int64

First 10 starts at ASSEMBLY_2
     case_id  timestamp          resource
20         0  24.866156  ASSEMBLY2_LINE_1
44         1  33.471312  ASSEMBLY2_LINE_2
64         3  36.252257  ASSEMBLY2_LINE_1
70         2  37.570474  ASSEMBLY2_LINE_2
90         6  44.083499  ASSEMBLY2_LINE_1
93         4  44.216695  ASSEMBLY2_LINE_2
124        5  57.923582  ASSEMBLY2_LINE_2
131        7  59.496962  ASSEMBLY2_LINE_1
137        8  60.244455  ASSEMBLY2_LINE_1
186       11  71.134149  ASSEMBLY2_LINE_2

Trace for case 0 at ASSEMBLY_1:
    timest

In [9]:
import pandas as pd
df = pd.read_csv(r'C:\Users\990215322\Desktop\MuProMAC\out\251110\results\FIFO_l0.28_actuator_manufacturing_with_rework.csv')
print(df['status'].value_counts())   # must include 'queued'

# starts at stations
starts = df[df["status"]=="running"][["activity","case_id","timestamp","resource"]] \
           .rename(columns={"timestamp":"start_time"})
# true station-arrivals
queued = df[df["status"]=="queued"][["activity","case_id","timestamp"]] \
           .rename(columns={"timestamp":"arrival_time"})

# visit-align per (activity, case_id)
starts["visit_idx"] = starts.groupby(["activity","case_id"]).cumcount()
queued["visit_idx"] = queued.groupby(["activity","case_id"]).cumcount()

visits = starts.merge(queued, on=["activity","case_id","visit_idx"], how="left")

rows, windows = [], []
for st, g in visits.groupby("activity", sort=True):
    g = g.sort_values("start_time").reset_index(drop=True)
    diffs = g["arrival_time"].diff()
    violations = int((diffs < 0).sum())
    rows.append((st, len(g), violations))
    # collect small windows around violations to inspect
    for i in g.index[diffs < 0][:5]:
        lo, hi = max(0, i-1), min(len(g)-1, i+1)
        windows.append(g.loc[lo:hi, ["activity","case_id","arrival_time","start_time","resource"]])

fifo = pd.DataFrame(rows, columns=["station","num_starts","fifo_violations"]).sort_values("station")
print(fifo)

if windows:
    print("\nWindows around violations:")
    print(pd.concat(windows, ignore_index=True))
else:
    print("\nNo FIFO breaches found using true queued times.")


status
queued      43781
running     43760
gateway     43755
START        8332
COMPLETE     8306
Name: count, dtype: int64
      station  num_starts  fifo_violations
0  ASSEMBLY_1        8754                0
1  ASSEMBLY_2        8747                0
2    MOULDING        8807                0
3   PACKAGING        8717                0
4     SORTING        8735                0

No FIFO breaches found using true queued times.


In [6]:
import pandas as pd
import numpy as np

# -----------------------------
# 0) Load event log
# -----------------------------
csv_path = "results/251022FIFO_l0.28_all_dedicated_sticky_with_rework.csv"  # <-- change if needed
df = pd.read_csv(csv_path)

# Keep only relevant cols
cols = ["simulation_run","timestamp","status","case_id","activity","resource","end_time","cycle_time","data","scenario","l","method"]
df = df[cols].copy()

# Parse numeric
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["end_time"] = pd.to_numeric(df["end_time"], errors="coerce")

# Basic schema sanity
required = ["simulation_run","timestamp","status","case_id","activity"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
assert df[required].isnull().sum().sum() == 0, "Nulls found in required fields"

# -----------------------------
# 1) Reconstruct task instances
#    (pair queued_k with running_k for each (sim, case, activity))
# -----------------------------
task_rows = df[df["activity"].ne("START") & df["status"].isin(["queued","running"])].copy()
task_rows = task_rows.sort_values(["simulation_run","case_id","activity","timestamp"]).copy()

queued = task_rows[task_rows["status"]=="queued"].copy()
queued["occ_q"] = queued.groupby(["simulation_run","case_id","activity"]).cumcount()

running = task_rows[task_rows["status"]=="running"].copy()
running["occ_r"] = running.groupby(["simulation_run","case_id","activity"]).cumcount()

inst = pd.merge(
    queued[["simulation_run","case_id","activity","occ_q","timestamp"]].rename(columns={"timestamp":"queued_ts"}),
    running[["simulation_run","case_id","activity","occ_r","timestamp","end_time","resource"]].rename(
        columns={"timestamp":"start_ts","end_time":"end_ts","resource":"resource_name"}),
    left_on=["simulation_run","case_id","activity","occ_q"],
    right_on=["simulation_run","case_id","activity","occ_r"],
    how="inner",
).drop(columns=["occ_q","occ_r"])

print("num task instances:", len(inst))
print("sample inst:\n", inst.head())

# Durations
inst["wait"]    = inst["start_ts"] - inst["queued_ts"]
inst["service"] = inst["end_ts"]   - inst["start_ts"]
inst["sojourn"] = inst["end_ts"]   - inst["queued_ts"]

neg = inst[(inst["wait"]<0) | (inst["service"]<0) | (inst["sojourn"]<0)]
assert neg.empty, f"Negative times found:\n{neg.head()}"

# -----------------------------
# 2) FIFO discipline per station
# -----------------------------
tmp = inst.sort_values(["simulation_run","activity","queued_ts","start_ts"]).copy()
eps = 1e-9
tmp["dec"] = tmp.groupby(["simulation_run","activity"])["start_ts"].diff() < -eps

fifo_check = (
    tmp.groupby(["simulation_run","activity"], as_index=False)
       .agg(num_instances=("start_ts","size"),
            fifo_violations=("dec","sum"))
)
fifo_check["fifo_violations"] = fifo_check["fifo_violations"].astype(int)

# -----------------------------
# 3) Lane-aware routing/QC audit
#    (dedicated+sticky scenario)
# -----------------------------
# Sort by (sim, case, time) and index within case
df_sorted = df.sort_values(["simulation_run","case_id", "timestamp"]).copy()
df_sorted["row_in_case"] = df_sorted.groupby(["simulation_run","case_id"]).cumcount()

# Gateway rows
gates = df_sorted[df_sorted["status"] == "gateway"][["simulation_run","case_id","timestamp","activity","row_in_case"]].copy()
gates = gates.sort_values(["simulation_run","case_id","row_in_case"]).reset_index(drop=True)

# Expected transitions for dedicated+sticky_with_rework
exp_rows = []

# QC after Moulding_i: rework -> MOULDING_i, pass -> ROUTE_TO_A1
for i in range(1, 6):
    g = f"QC_AFTER_MOULDING_{i}"
    exp_rows += [
        {"activity": g, "next_activity": f"MOULDING_{i}",   "expected": 0.05},
        {"activity": g, "next_activity": "ROUTE_TO_A1",     "expected": 0.95},
    ]

# QC after A1_i: rework -> ASSEMBLY_1_i, pass -> ROUTE_TO_A2
for i in range(1, 6):
    g = f"QC_AFTER_A1_{i}"
    exp_rows += [
        {"activity": g, "next_activity": f"ASSEMBLY_1_{i}", "expected": 0.05},
        {"activity": g, "next_activity": "ROUTE_TO_A2",     "expected": 0.95},
    ]

# QC after A2_i: rework -> ASSEMBLY_2_i, pass -> ROUTE_TO_SORTING
for i in (1, 2):
    g = f"QC_AFTER_A2_{i}"
    exp_rows += [
        {"activity": g, "next_activity": f"ASSEMBLY_2_{i}", "expected": 0.05},
        {"activity": g, "next_activity": "ROUTE_TO_SORTING","expected": 0.95},
    ]

# QC after Sorting (single): rework -> SORTING, pass -> ROUTE_TO_PACKAGING
exp_rows += [
    {"activity": "QC_AFTER_SORTING", "next_activity": "SORTING",             "expected": 0.05},
    {"activity": "QC_AFTER_SORTING", "next_activity": "ROUTE_TO_PACKAGING",  "expected": 0.95},
]

# QC after Packaging_i: rework -> PACKAGING_i, pass -> END
for i in (1, 2, 3):
    g = f"QC_AFTER_PACKAGING_{i}"
    exp_rows += [
        {"activity": g, "next_activity": f"PACKAGING_{i}", "expected": 0.05},
        {"activity": g, "next_activity": "END",            "expected": 0.95},
    ]

routing_exp = pd.DataFrame(exp_rows)

# Next-node extractor: first row after gateway (router OR task OR case end)
def next_node_after_gateway(sim_run, case_id, row_idx_after):
    sub = df_sorted[
        (df_sorted["simulation_run"] == sim_run) &
        (df_sorted["case_id"] == case_id) &
        (df_sorted["row_in_case"] > row_idx_after)
    ]
    if sub.empty:
        return np.nan
    r = sub.iloc[0]
    if r["status"] == "COMPLETE":
        return "END"
    return r["activity"]

gates["next_activity"] = [
    next_node_after_gateway(sim_run, cid, r)
    for sim_run, cid, r in gates[["simulation_run","case_id","row_in_case"]]
        .itertuples(index=False, name=None)
]

routing_emp = (
    gates.groupby("activity")["next_activity"]
         .value_counts(normalize=True, dropna=True)
         .rename("prob")
         .reset_index()
)

routing = (routing_exp
           .merge(routing_emp, on=["activity","next_activity"], how="left")
           .fillna({"prob": 0.0}))
routing["diff"] = routing["prob"] - routing["expected"]

print("\nRouting empirical vs expected (lane-aware):")
print(routing.sort_values(["activity","next_activity"]))

# Zero-wait share from gateway to the first queued task (skipping routers)
def first_queued_after_gateway(sim_run, case_id, row_idx_after):
    sub = df_sorted[
        (df_sorted["simulation_run"] == sim_run) &
        (df_sorted["case_id"] == case_id) &
        (df_sorted["row_in_case"] > row_idx_after)
    ]
    if sub.empty:
        return np.nan
    nq = sub.loc[sub["status"] == "queued"]
    return nq.iloc[0]["timestamp"] if not nq.empty else np.nan

gates["first_queued_ts"] = [
    first_queued_after_gateway(sim_run, cid, r)
    for sim_run, cid, r in gates[["simulation_run","case_id","row_in_case"]]
        .itertuples(index=False, name=None)
]
zero_wait_share = (gates["first_queued_ts"] - gates["timestamp"] <= 1e-9).mean()
print("Share of gateways with immediate (zero-wait) handoff:", zero_wait_share)

# Optional sanity
missing_gates = set(routing_exp["activity"]) - set(gates["activity"])
if missing_gates:
    print("Expected gateways missing in log:", sorted(missing_gates))
unknown_gates = set(gates["activity"]) - set(routing_exp["activity"])
if unknown_gates:
    print("Gateways present but not in expected map (FYI):", sorted(unknown_gates))

# -----------------------------
# 4) Service-time by resource
# -----------------------------
svc = inst.dropna(subset=["resource_name","service"]).copy()
svc_stats = svc.groupby("resource_name")["service"].agg(["count","mean","std","min","max"]).reset_index()
svc_stats["std_over_mean"] = svc_stats["std"]/svc_stats["mean"]

# -----------------------------
# 5) Utilization & throughput
# -----------------------------
T_per_run = df.groupby("simulation_run")["timestamp"].max()
T = T_per_run.mean()  # average horizon

busy = (
    svc.groupby("resource_name", as_index=False)["service"]
       .sum()
       .rename(columns={"service":"busy_time"})
)
busy["utilization"] = busy["busy_time"] / (T * len(T_per_run))

# Station KPIs
station = inst.groupby("activity").agg(
    arrivals=("activity","size"),
    mean_wait=("wait","mean"),
    mean_service=("service","mean"),
    mean_sojourn=("sojourn","mean")
).reset_index()
station["lambda"] = station["arrivals"] / (T * len(T_per_run))
station["L_hat"] = station["lambda"] * station["mean_sojourn"]

print("\nFIFO violations per station per run (should be 0):")
print(fifo_check.groupby("activity")["fifo_violations"].sum().sort_values(ascending=False).head(10))

print("\nService by resource (std/mean ~ 1 for Exp):")
print(svc_stats.sort_values("std_over_mean"))

print("\nResource utilization (sanity):")
print(busy.sort_values("utilization", ascending=False).head(10))

print("\nStation KPIs + Little's Law proxy:")
print(station.sort_values("L_hat", ascending=False).head(10))

# -----------------------------
# 6) Stability audit (warm-up trimmed)
# -----------------------------
warmup_fraction = 0.05

def analyze_stability_for_run(run_df, run_id):
    warmup = run_df["timestamp"].max() * warmup_fraction
    inst_run = inst[inst["simulation_run"] == run_id].copy()
    inst_ss = inst_run[inst_run["queued_ts"] >= warmup].copy()
    if inst_ss.empty:
        return None

    T_eff = run_df["timestamp"].max() - warmup
    arrivals = inst_ss.groupby("activity")["activity"].size().rename("arrivals").reset_index()
    arrivals["lambda_station"] = arrivals["arrivals"] / T_eff

    svc_ss = inst_ss.dropna(subset=["resource_name","service"]).copy()
    if svc_ss.empty:
        return None

    rs = (svc_ss.groupby(["activity","resource_name"])["service"]
                .mean()
                .rename("mean_service")
                .reset_index())
    cap = (rs.assign(mu=lambda d: 1.0 / d["mean_service"])
             .groupby("activity")["mu"].sum()
             .rename("capacity")
             .reset_index())

    stab = (arrivals.merge(cap, on="activity", how="left"))
    stab["rho_hat"] = stab["lambda_station"] / stab["capacity"]
    stab["headroom"] = 1.0 - stab["rho_hat"]
    stab["simulation_run"] = run_id
    return stab

stability_results = []
for run_id in df["simulation_run"].unique():
    run_df = df[df["simulation_run"] == run_id]
    stab = analyze_stability_for_run(run_df, run_id)
    if stab is not None:
        stability_results.append(stab)

if stability_results:
    combined_stability = pd.concat(stability_results, ignore_index=True)
    avg_stability = combined_stability.groupby("activity").agg({
        "rho_hat": "mean",
        "headroom": "mean",
        "lambda_station": "mean",
        "capacity": "mean"
    }).reset_index()
    print("\nAverage stability headroom by station across all runs:")
    print(avg_stability.sort_values("rho_hat", ascending=False))
else:
    print("\nNo stability data available")

# -----------------------------
# 7) Queue trend analysis
# -----------------------------
def queue_trace(inst_df, act, run_id):
    run_data = inst_df[inst_df["simulation_run"] == run_id]
    g = run_data[run_data["activity"]==act][["queued_ts","start_ts"]].copy()
    if g.empty:
        return None
    ups   = g.groupby("queued_ts").size().rename("delta")
    downs = (g.groupby("start_ts").size() * -1).rename("delta")
    deltas = pd.concat([ups, downs], axis=0).groupby(level=0).sum().sort_index()
    tr = deltas.cumsum().reset_index().rename(columns={"index":"time","delta":"qlen"})
    tr["qlen"] = tr["qlen"].clip(lower=0)
    return tr

def end_trend_slope(trace, tail_frac=0.3):
    if trace is None or len(trace) < 5:
        return np.nan
    n0 = int(len(trace)*(1-tail_frac))
    tail = trace.iloc[max(n0,0):].copy()
    if len(tail) < 3:
        return np.nan
    x = tail["time"].values
    y = tail["qlen"].values
    A = np.vstack([x, np.ones_like(x)]).T
    slope, _ = np.linalg.lstsq(A, y, rcond=None)[0]
    return slope

trend_results = []
for run_id in df["simulation_run"].unique():
    run_inst = inst[inst["simulation_run"] == run_id]
    warmup = run_inst["queued_ts"].max() * warmup_fraction if not run_inst.empty else 0.0
    inst_ss_run = run_inst[run_inst["queued_ts"] >= warmup]
    acts = sorted(inst_ss_run["activity"].unique())
    for act in acts:
        tr = queue_trace(inst_ss_run, act, run_id)
        slope = end_trend_slope(tr, tail_frac=0.3)
        trend_results.append({"simulation_run": run_id, "activity": act, "end_slope_q_per_min": slope})

trend_df = pd.DataFrame(trend_results)
avg_trend = trend_df.groupby("activity")["end_slope_q_per_min"].mean().reset_index()

print("\nAverage queue end-trend slope across all runs (jobs/min; ≈0 is good):")
print(avg_trend.sort_values("end_slope_q_per_min", ascending=False))


num task instances: 44740
sample inst:
    simulation_run  case_id      activity  queued_ts   start_ts     end_ts  \
0               0        0  ASSEMBLY_1_2  31.440372  31.440372  32.924647   
1               0        0  ASSEMBLY_2_1  32.924647  32.924647  33.427845   
2               0        0    MOULDING_2  18.537263  18.537263  25.080815   
3               0        0    MOULDING_2  25.080815  25.080815  31.440372   
4               0        0   PACKAGING_3  43.575669  43.575669  50.967978   

        resource_name  
0    LINE_2_ASSEMBLY1  
1    ASSEMBLY2_LINE_1  
2  MOULDING_MACHINE_2  
3  MOULDING_MACHINE_2  
4    PACKAGING_LINE_3  

Routing empirical vs expected (lane-aware):
                activity       next_activity  expected      prob      diff
10         QC_AFTER_A1_1        ASSEMBLY_1_1      0.05  0.055435  0.005435
11         QC_AFTER_A1_1         ROUTE_TO_A2      0.95  0.944565 -0.005435
12         QC_AFTER_A1_2        ASSEMBLY_1_2      0.05  0.065366  0.015366
13      

In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\990215322\Desktop\MuProMAC\results\251022FIFO_l0.28_all_dedicated_sticky_with_rework.csv")

mask = df["data"].astype(str).str.contains("mould_lane|a1_lane|a2_lane|pack_lane", na=False)
print("Rows with lane keys:", mask.sum(), "out of", len(df))

# where do they appear?
print(df.loc[mask, "status"].value_counts())

# peek at a few rows
print(df.loc[mask, ["timestamp","case_id","status","activity","data"]].head(10).to_string(index=False))


Rows with lane keys: 67313 out of 177362
status
gateway    67313
Name: count, dtype: int64
 timestamp  case_id  status            activity                         data
 22.457367        1 gateway QC_AFTER_MOULDING_4 {'mould_lane': 'MOULDING_4'}
 22.457367        1 gateway         ROUTE_TO_A1 {'mould_lane': 'MOULDING_4'}
 25.080815        0 gateway QC_AFTER_MOULDING_2 {'mould_lane': 'MOULDING_2'}
 25.080815        0 gateway         ROUTE_TO_A1 {'mould_lane': 'MOULDING_2'}
 25.151521        3 gateway QC_AFTER_MOULDING_2 {'mould_lane': 'MOULDING_2'}
 25.151521        3 gateway         ROUTE_TO_A1 {'mould_lane': 'MOULDING_2'}
 26.001849        2 gateway QC_AFTER_MOULDING_3 {'mould_lane': 'MOULDING_3'}
 26.001849        2 gateway         ROUTE_TO_A1 {'mould_lane': 'MOULDING_3'}
 26.994714        1 gateway       QC_AFTER_A1_5 {'mould_lane': 'MOULDING_4'}
 26.994714        1 gateway         ROUTE_TO_A2 {'mould_lane': 'MOULDING_4'}


In [10]:
import pandas as pd
import numpy as np

# --- Load ---
df = pd.read_csv(r"C:\Users\990215322\Desktop\MuProMAC\out\251110\results\FIFO_l0.28_actuator_manufacturing_with_rework.csv")

# Keep only relevant cols
cols = ["simulation_run","timestamp","status","case_id","activity","resource","end_time","cycle_time","data","scenario","l","method"]
df = df[cols].copy()

# Parse numeric
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["end_time"]   = pd.to_numeric(df["end_time"],   errors="coerce")

# --- 1) basic schema sanity ---
required = ["simulation_run","timestamp","status","case_id","activity"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
assert df[required].isnull().sum().sum() == 0, "Nulls found in required fields"

# --- 2) reconstruct task instances (robust pairing) ---
# Keep only task lifecycle rows (exclude gateways and START/COMPLETE for case)
task_rows = df[df["activity"].ne("START") & df["status"].isin(["queued","running"])].copy()

# Sort so cumcounts respect time
task_rows = task_rows.sort_values(["simulation_run","case_id","activity","timestamp"]).copy()

# Split and make occurrence indices *per status* (not mixed)
queued = task_rows[task_rows["status"]=="queued"].copy()
queued["occ_q"] = queued.groupby(["simulation_run","case_id","activity"]).cumcount()

running = task_rows[task_rows["status"]=="running"].copy()
running["occ_r"] = running.groupby(["simulation_run","case_id","activity"]).cumcount()

# Merge kth queued with kth running for each (simulation_run, case, activity)
inst = pd.merge(
    queued[["simulation_run","case_id","activity","occ_q","timestamp"]].rename(columns={"timestamp":"queued_ts"}),
    running[["simulation_run","case_id","activity","occ_r","timestamp","end_time","resource"]].rename(
        columns={"timestamp":"start_ts","end_time":"end_ts","resource":"resource_name"}),
    left_on=["simulation_run","case_id","activity","occ_q"],
    right_on=["simulation_run","case_id","activity","occ_r"],
    how="inner",
).drop(columns=["occ_q","occ_r"])

print("num task instances:", len(inst))
print("sample inst:\n", inst.head())

# Durations
inst["wait"]    = inst["start_ts"] - inst["queued_ts"]
inst["service"] = inst["end_ts"]   - inst["start_ts"]
inst["sojourn"] = inst["end_ts"]   - inst["queued_ts"]

# Sanity: no negatives
neg = inst[(inst["wait"]<0) | (inst["service"]<0) | (inst["sojourn"]<0)]
assert neg.empty, f"Negative times found:\n{neg.head()}"

# --- 3) FIFO discipline per station (warning-free) ---
tmp = inst.sort_values(["simulation_run","activity","queued_ts","start_ts"]).copy()
eps = 1e-9
tmp["dec"] = tmp.groupby(["simulation_run","activity"])["start_ts"].diff() < -eps

fifo_check = (
    tmp.groupby(["simulation_run","activity"], as_index=False)
       .agg(num_instances=("start_ts","size"),
            fifo_violations=("dec","sum"))
)
fifo_check["fifo_violations"] = fifo_check["fifo_violations"].astype(int)

# --- 4) routing/QC probabilities (guarded for no-gateway logs) ---
df_sorted = df.sort_values(["simulation_run","case_id","timestamp"]).copy()
df_sorted["row_in_case"] = df_sorted.groupby(["simulation_run","case_id"]).cumcount()

gates = (df_sorted[df_sorted["status"] == "gateway"]
         [["simulation_run","case_id","timestamp","activity","row_in_case"]]
         .sort_values(["simulation_run","case_id","row_in_case"])
         .reset_index(drop=True))

if gates.empty:
    print("\nNo gateways found in this log (as expected for the *no_rework* scenario).")
    routing_emp = pd.DataFrame(columns=["activity","next_activity","prob"])
    zero_wait_share = np.nan
else:
    def next_step_after_gateway(sim_run, case_id, row_idx_after):
        sub = df_sorted[(df_sorted["simulation_run"] == sim_run) &
                        (df_sorted["case_id"] == case_id) &
                        (df_sorted["row_in_case"] > row_idx_after)]
        if sub.empty:
            return np.nan
        nxt_q = sub.loc[sub["status"] == "queued"]
        if not nxt_q.empty:
            return nxt_q.iloc[0]["activity"]
        nxt_c = sub.loc[sub["status"] == "COMPLETE"]
        if not nxt_c.empty:
            return "END"
        return np.nan

    gates["next_activity"] = [
        next_step_after_gateway(sim_run, cid, r)
        for sim_run, cid, r in gates[["simulation_run","case_id","row_in_case"]].itertuples(index=False, name=None)
    ]

    routing_emp = (
        gates.groupby("activity")["next_activity"]
             .value_counts(normalize=True, dropna=True)
             .rename("prob")
             .reset_index()
    )

    merge_for_wait = gates.merge(
        inst[["simulation_run","case_id","activity","queued_ts","start_ts"]],
        left_on=["simulation_run","case_id","next_activity"],
        right_on=["simulation_run","case_id","activity"],
        how="left", suffixes=("_gate","_next")
    )
    zero_wait_share = (merge_for_wait["start_ts"] - merge_for_wait["timestamp"] <= 1e-9).mean()

print("Share of gateways with immediate (zero-wait) handoff:", zero_wait_share)

# (No expected-vs-empirical table shown because no gateways in this scenario.)

# --- 5) service-time by resource ---
svc = inst.dropna(subset=["resource_name","service"]).copy()
svc_stats = (svc.groupby("resource_name")["service"]
               .agg(["count","mean","std","min","max"])
               .reset_index())
svc_stats["std_over_mean"] = svc_stats["std"] / svc_stats["mean"]

# --- 6) utilization & throughput ---
# Average T across runs (works for 1 run too)
T_per_run = df.groupby("simulation_run")["timestamp"].max()
T = T_per_run.mean()

busy = (svc.groupby("resource_name", as_index=False)["service"]
          .sum()
          .rename(columns={"service":"busy_time"}))
busy["utilization"] = busy["busy_time"] / (T * len(T_per_run))

# Station-level KPIs
station = (inst.groupby("activity")
              .agg(arrivals=("activity","size"),
                   mean_wait=("wait","mean"),
                   mean_service=("service","mean"),
                   mean_sojourn=("sojourn","mean"))
              .reset_index())

# Little's Law proxy
station["lambda"] = station["arrivals"] / (T * len(T_per_run))
station["L_hat"]  = station["lambda"] * station["mean_sojourn"]

# --- Summaries ---
print("\nFIFO violations per station per run (should be 0):")
print(fifo_check.groupby("activity")["fifo_violations"].sum().sort_values(ascending=False).head(10))

print("\nService by resource (std/mean ~ 1 for Exp):")
print(svc_stats.sort_values("std_over_mean"))

print("\nResource utilization (sanity):")
print(busy.sort_values("utilization", ascending=False).head(10))

print("\nStation KPIs + Little's Law proxy:")
print(station.sort_values("L_hat", ascending=False).head(10))

# --- 7) Stability audit (per run with warm-up), then average ---
warmup_fraction = 0.05

def analyze_stability_for_run(run_df, inst_df, run_id):
    warmup = run_df["timestamp"].max() * warmup_fraction
    inst_ss = inst_df[inst_df["queued_ts"] >= warmup].copy()
    if inst_ss.empty:
        return None
    T_eff = run_df["timestamp"].max() - warmup

    arrivals = (inst_ss.groupby("activity")["activity"]
                        .size().rename("arrivals").reset_index())
    arrivals["lambda_station"] = arrivals["arrivals"] / T_eff

    svc_ss = inst_ss.dropna(subset=["resource_name","service"]).copy()
    if svc_ss.empty:
        return None
    rs = (svc_ss.groupby(["activity","resource_name"])["service"]
                .mean().rename("mean_service").reset_index())
    cap = (rs.assign(mu=lambda d: 1.0 / d["mean_service"])
             .groupby("activity")["mu"].sum().rename("capacity").reset_index())

    stab = arrivals.merge(cap, on="activity", how="left")
    stab["rho_hat"]   = stab["lambda_station"] / stab["capacity"]
    stab["headroom"]  = 1.0 - stab["rho_hat"]
    stab["simulation_run"] = run_id
    return stab

stability_results = []
for run_id in df["simulation_run"].unique():
    run_df  = df[df["simulation_run"] == run_id]
    inst_df = inst[inst["simulation_run"] == run_id]
    stab = analyze_stability_for_run(run_df, inst_df, run_id)
    if stab is not None:
        stability_results.append(stab)

if stability_results:
    combined_stability = pd.concat(stability_results, ignore_index=True)
    avg_stability = (combined_stability.groupby("activity")
                     .agg(rho_hat=("rho_hat","mean"),
                          headroom=("headroom","mean"),
                          lambda_station=("lambda_station","mean"),
                          capacity=("capacity","mean"))
                     .reset_index())
    print("\nAverage stability headroom by station across all runs:")
    print(avg_stability.sort_values("rho_hat", ascending=False))
else:
    print("\nNo stability data available")

# --- 8) Queue trend analysis ---
def queue_trace(inst_df, act, run_id):
    run_data = inst_df[inst_df["simulation_run"] == run_id]
    g = run_data[run_data["activity"]==act][["queued_ts","start_ts"]].copy()
    if g.empty:
        return None
    ups   = g.groupby("queued_ts").size().rename("delta")
    downs = (g.groupby("start_ts").size() * -1).rename("delta")
    deltas = pd.concat([ups, downs], axis=0).groupby(level=0).sum().sort_index()
    tr = deltas.cumsum().reset_index().rename(columns={"index":"time","delta":"qlen"})
    tr["qlen"] = tr["qlen"].clip(lower=0)
    return tr

def end_trend_slope(trace, tail_frac=0.2):
    if trace is None or len(trace) < 5:
        return np.nan
    n0 = int(len(trace)*(1-tail_frac))
    tail = trace.iloc[max(n0,0):].copy()
    if len(tail) < 3:
        return np.nan
    x = tail["time"].values
    y = tail["qlen"].values
    A = np.vstack([x, np.ones_like(x)]).T
    slope, _ = np.linalg.lstsq(A, y, rcond=None)[0]
    return slope

trend_results = []
for run_id in df["simulation_run"].unique():
    inst_run = inst[inst["simulation_run"] == run_id]
    warmup = inst_run["queued_ts"].max() * warmup_fraction if not inst_run.empty else 0.0
    inst_ss_run = inst_run[inst_run["queued_ts"] >= warmup]

    for act in sorted(inst_ss_run["activity"].unique()):
        tr = queue_trace(inst_ss_run, act, run_id)
        slope = end_trend_slope(tr, tail_frac=0.3)
        trend_results.append({"simulation_run": run_id, "activity": act, "end_slope_q_per_min": slope})

trend_df = pd.DataFrame(trend_results)
avg_trend = trend_df.groupby("activity")["end_slope_q_per_min"].mean().reset_index()

print("\nAverage queue end-trend slope across all runs (jobs/min; ≈0 is good):")
print(avg_trend.sort_values("end_slope_q_per_min", ascending=False))


num task instances: 43760
sample inst:
    simulation_run  case_id    activity  queued_ts   start_ts     end_ts  \
0               0        0  ASSEMBLY_1  20.393651  20.393651  24.866156   
1               0        0  ASSEMBLY_2  24.866156  24.866156  26.676780   
2               0        0    MOULDING  18.537263  18.537263  20.393651   
3               0        0   PACKAGING  29.063644  29.063644  36.031979   
4               0        0     SORTING  26.676780  26.676780  29.063644   

        resource_name  
0    LINE_1_ASSEMBLY1  
1    ASSEMBLY2_LINE_1  
2  MOULDING_MACHINE_1  
3    PACKAGING_LINE_1  
4       SORTING_ROBOT  
Share of gateways with immediate (zero-wait) handoff: 0.4424941491140087

FIFO violations per station per run (should be 0):
activity
ASSEMBLY_1    0
ASSEMBLY_2    0
MOULDING      0
PACKAGING     0
SORTING       0
Name: fifo_violations, dtype: int64

Service by resource (std/mean ~ 1 for Exp):
         resource_name  count      mean       std       min        max

In [ ]:
import pandas as pd
import pm4py

def import_csv(file_path):
    # 1) Try autodetecting the delimiter (requires engine="python"; don't pass low_memory)
    df = pd.read_csv(file_path, sep=None, engine="python", dtype=str, encoding="utf-8-sig")

    # 2) If it still came in as a single column, try common delimiters
    if df.shape[1] == 1:
        for sep in [",", ";", "\t", "|"]:
            df2 = pd.read_csv(file_path, sep=sep, dtype=str, encoding="utf-8-sig")
            if df2.shape[1] > 1:
                df = df2
                break

    # 3) Clean headers (strip & remove BOM just in case)
    df.columns = (df.columns
                    .str.replace(r'^\ufeff', '', regex=True)
                    .str.strip())

    # 4) Show what we’ve got
    print(f"Shape: {df.shape}")
    print("Columns:", list(df.columns))

    # 5) Pick a case-id-like column robustly
    candidates = ['case_id', 'case:concept:name', 'case id', 'caseid', 'case']
    case_col = None

    # exact (case-insensitive)
    for k in candidates:
        hits = [c for c in df.columns if c.lower() == k]
        if hits:
            case_col = hits[0]
            break
    # fuzzy fallback
    if case_col is None:
        for c in df.columns:
            low = c.lower()
            if 'case' in low and ('id' in low or 'concept:name' in low):
                case_col = c
                break

    if case_col is None:
        print("Could not find a case-id column. Inspect the headers above.")
        return

    num_events = len(df)
    num_cases = df[case_col].nunique(dropna=True)

    print(f"Using case column: '{case_col}'")
    print(f"Number of events: {num_events}\nNumber of cases: {num_cases}")

df = pm4py.format_dataframe(df, case_id = 'case_id', activity_key='activity'; timestamp_key='timestamp', timest_format= )
start_activities = pm4py.get_start_activities(df)
end_activities = pm4py.get_end_activities(df)
print("Start activities: {}\nEnd activities: {}".format(start_activities, end_activities))

# Run
import_csv(r"C:\Users\990215322\Desktop\MuProMAC\results\251021FIFO_l0.28_actuator_manufacturing_with_rework.csv")


In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\990215322\Desktop\MuProMAC\results\251022FIFO_l0.28_all_dedicated_sticky_with_rework.csv")

mask = df["data"].astype(str).str.contains("mould_lane|a1_lane|a2_lane|pack_lane", na=False)
print("Rows with lane keys:", mask.sum(), "out of", len(df))

# where do they appear?
print(df.loc[mask, "status"].value_counts())

# peek at a few rows
print(df.loc[mask, ["timestamp","case_id","status","activity","data"]].head(10).to_string(index=False))


In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\990215322\Desktop\MuProMAC\results\251022FIFO_l0.28_all_dedicated_sticky_with_rework.csv")

s = df["data"].astype(str)
print("mould_lane rows:", s.str.contains(r"\bmould_lane\b", na=False).sum())
print("a1_lane rows   :", s.str.contains(r"\ba1_lane\b",    na=False).sum())
print("a2_lane rows   :", s.str.contains(r"\ba2_lane\b",    na=False).sum())
print("pack_lane rows :", s.str.contains(r"\bpack_lane\b",  na=False).sum())

# show a few places where a1_lane appears (should be around QC_AFTER_A1_* or ROUTE_TO_A2)
mask_a1 = s.str.contains(r"\ba1_lane\b", na=False)
print(df.loc[mask_a1, ["timestamp","case_id","status","activity","data"]].head(10).to_string(index=False))

# same for a2 and packaging
mask_a2 = s.str.contains(r"\ba2_lane\b", na=False)
mask_pk = s.str.contains(r"\bpack_lane\b", na=False)
print(df.loc[mask_a2, ["timestamp","case_id","status","activity","data"]].head(10).to_string(index=False))
print(df.loc[mask_pk, ["timestamp","case_id","status","activity","data"]].head(10).to_string(index=False))
